# 02. Data Loading and Preparation

This notebook validates the MUSAE edge list, targets, and sparse content features, then constructs the undirected mutual-follow graph used by the remaining notebooks.


In [1]:
from pathlib import Path
import os
import sys
import warnings

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path) -> Path:
    """Find the repository root whether Jupyter starts there or in notebooks/."""
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root.")


ROOT = find_project_root(Path.cwd())
os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
print(f"Project root: {ROOT}")


Project root: C:\Users\Jimi\Desktop\github-social-network


## Essential cleaning and graph-construction logic

Each edge is an undirected mutual-follow relation. The reusable loader in `src/data_loader.py` converts IDs to integers and raises an error for missing endpoints, self-loops, duplicate undirected edges, duplicate target IDs, invalid binary labels, or graph nodes without labels. The essential logic is:

```text
read edges and targets
require the expected columns and numeric IDs
reject missing values and self-loops
canonicalize every pair as (min(u,v), max(u,v)) and reject duplicates
require one binary label per node
G = undirected graph from the validated edge list
require every graph node to have a target
```


In [2]:
from content_features import load_content_features
from data_loader import load_dataset

G, edges_df, labels_df, labels_dict = load_dataset(ROOT / "data")
content_dict, n_content_features = load_content_features(
    ROOT / "data" / "musae_git_features.json"
)

canonical_edges = pd.DataFrame({
    "u": edges_df[["id_1", "id_2"]].min(axis=1),
    "v": edges_df[["id_1", "id_2"]].max(axis=1),
})
validation = pd.DataFrame([{
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "connected_components": nx.number_connected_components(G),
    "self_loops": nx.number_of_selfloops(G),
    "duplicate_undirected_edges": int(canonical_edges.duplicated().sum()),
    "labeled_nodes": len(labels_dict),
    "content_nodes": len(content_dict),
    "content_feature_dimensions": n_content_features,
}])
display(validation)
display(edges_df.head())
display(labels_df.head())

assert validation.iloc[0].to_dict() == {
    "nodes": 37700,
    "edges": 289003,
    "connected_components": 1,
    "self_loops": 0,
    "duplicate_undirected_edges": 0,
    "labeled_nodes": 37700,
    "content_nodes": 37700,
    "content_feature_dimensions": 4005,
}
assert set(G) == set(labels_dict) == set(content_dict)


,nodes,edges,connected_components,self_loops,duplicate_undirected_edges,labeled_nodes,content_nodes,content_feature_dimensions
0,37700,289003,1,0,0,37700,37700,4005


,id_1,id_2
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


,id,name,label
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


The validated graph has 37,700 labeled nodes and 289,003 edges in one connected component. All 37,700 nodes also occur in the content-feature mapping. The cleaning rules reject invalid input instead of silently dropping or repairing rows, so a malformed dataset cannot change the graph unnoticed.

## Summary

The graph, labels, and content-feature IDs are aligned. Later notebooks reload these objects themselves; they do not depend on this kernel.
